# exp131_gr_shape_descriptor_matching_ablation train

Target-free GR shape descriptor matching ablation for existing PF/Beam/likelihood-PF candidates. This notebook saves descriptor score diagnostics and an exp072-style wide train feature cache for downstream verifier experiments.


## Contents

1. Setup and configuration
2. Input and audit/cache contract
3. Run descriptor matching ablation
4. Preview outputs
5. Metrics and next branch


## 1. Setup and configuration


In [ ]:
from __future__ import annotations

import json
import os
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

from settings import EXPERIMENT_NAME, ExperimentPaths, get_nested, load_config
from gr_shape_descriptor_matching_ablation import run_from_config, to_jsonable

DEBUG = os.environ.get("EXPERIMENT_DEBUG", "0") == "1"
paths = ExperimentPaths()
paths.ensure_output_dirs()
config = load_config()

if DEBUG:
    config.setdefault("audit", {})["max_rows"] = int(os.environ.get("EXPERIMENT_DEBUG_MAX_ROWS", "20000"))

print(json.dumps({
    "experiment": EXPERIMENT_NAME,
    "route": get_nested(config, "experiment.route"),
    "status": get_nested(config, "experiment.status"),
    "parent": get_nested(config, "lineage.parent"),
    "cache_parent": get_nested(config, "lineage.cache_parent"),
    "debug": DEBUG,
    "max_rows": get_nested(config, "audit.max_rows"),
    "train_data_dir": str(paths.train_data_dir),
    "artifacts_dir": str(paths.artifacts_dir),
}, indent=2, ensure_ascii=False))


## 2. Input and audit/cache contract


In [ ]:
print(json.dumps({
    "descriptor_matching": get_nested(config, "model.descriptor_matching"),
    "score_variants": get_nested(config, "audit.score_variants"),
    "candidate_sets": get_nested(config, "audit.candidate_sets"),
    "thresholds_ft": get_nested(config, "audit.thresholds_ft"),
    "topk_values": get_nested(config, "audit.topk_values"),
    "leakage_policy": get_nested(config, "validation.leakage_policy"),
    "expected_train_artifacts": get_nested(config, "audit.expected_train_artifacts"),
}, indent=2, ensure_ascii=False))


## 3. Run descriptor matching ablation


In [ ]:
summary = run_from_config(config)
print(json.dumps(to_jsonable({
    "status": summary["status"],
    "runtime_seconds": summary["runtime_seconds"],
    "rows": summary["source"]["rows"],
    "wells": summary["source"]["wells"],
    "score_variants": summary["descriptor_matching"]["score_variants"],
    "train_feature_cache": {
        "variant": summary["train_feature_cache"]["variant"],
        "feature_count": summary["train_feature_cache"]["feature_count"],
        "rows": summary["train_feature_cache"]["rows"],
        "sha256": summary["train_feature_cache"]["sha256"],
    },
    "best_candidate_by_rmse": summary["best_candidate_by_rmse"],
    "best_score_variant_by_auc": summary["best_score_variant_by_auc"],
    "probe_decision": summary["probe_decision"],
}), indent=2, ensure_ascii=False))


## 4. Preview outputs


In [ ]:
artifact_paths = {name: paths.artifacts_dir / filename for name, filename in summary["outputs"].items() if filename}
for name, path in artifact_paths.items():
    print(f"{name}: {path} exists={path.exists()} size={path.stat().st_size if path.exists() else 0}")

def preview_csv(name: str, n: int = 10) -> pd.DataFrame:
    path = artifact_paths[name]
    frame = pd.read_csv(path, nrows=n)
    display(frame)
    return frame

candidate_metrics_preview = preview_csv("candidate_metrics")
score_variant_metrics_preview = preview_csv("score_variant_metrics")
rank_metrics_preview = preview_csv("rank_metrics")
bucket_metrics_preview = preview_csv("bucket_metrics")
by_well_preview = preview_csv("by_well")


## 5. Metrics and next branch


In [ ]:
metrics = {
    "experiment": EXPERIMENT_NAME,
    "status": summary["status"],
    "updated_at": datetime.now(timezone.utc).isoformat(),
    "route": get_nested(config, "experiment.route"),
    "metric": "candidate_auc_logloss_topk",
    "source": summary["source"],
    "descriptor_matching": summary["descriptor_matching"],
    "train_feature_cache": summary["train_feature_cache"],
    "best_candidate_by_rmse": summary["best_candidate_by_rmse"],
    "best_score_variant_by_auc": summary["best_score_variant_by_auc"],
    "probe_decision": summary["probe_decision"],
    "prediction_sha": summary["prediction_sha"],
    "outputs": summary["outputs"],
}
metrics_path = paths.experiment_dir / "metrics.json"
metrics_path.write_text(json.dumps(to_jsonable(metrics), indent=2, sort_keys=True), encoding="utf-8")
print(metrics_path)
print(json.dumps(to_jsonable(metrics["probe_decision"]), indent=2, ensure_ascii=False))
